In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


subscr_dates = (
    subscr_status
    .groupby('magnit_id', as_index=False)
    .agg(first_subscr_date=('subscr_date_act', 'min'))
)

clicks = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    [['magnit_id', 'calc_date']]
    .drop_duplicates()
)

clicks['calc_date'] = pd.to_datetime(clicks['calc_date'])
subscr_dates['first_subscr_date'] = pd.to_datetime(subscr_dates['first_subscr_date'])

clicks = clicks.merge(subscr_dates, on='magnit_id', how='left')

clicks['click_without_subscr'] = (
    clicks['first_subscr_date'].isna()
    | (clicks['calc_date'] < clicks['first_subscr_date'])
)

click_clients = (
    clicks[clicks['click_without_subscr']]
    [['magnit_id']]
    .drop_duplicates()
)

click_clients['has_click_without_subscr'] = 1

result = (
    client_cohorts
    .merge(click_clients, on='magnit_id', how='left')
)

result['has_click_without_subscr'] = (
    result['has_click_without_subscr']
    .fillna(0)
)

cohort_clicks = (
    result
    .groupby('campaigns_cnt', as_index=False)
    .agg(
        client_cnt=('client_id', 'nunique'),
        click_cnt=('has_click_without_subscr', 'sum')
    )
)

cohort_clicks['click_pct'] = (
    cohort_clicks['click_cnt']
    / cohort_clicks['client_cnt']
    * 100
).round(2)

In [ ]:
cohort_clicks_table = (
    cohort_clicks
    .rename(
        columns={
            'campaigns_cnt': 'Количество кампаний',
            'client_cnt': 'Количество клиентов',
            'click_cnt': 'Клиенты с просмотрами/кликами без подписки',
            'click_pct': 'Доля клиентов, %'
        }
    )
)

display(
    cohort_clicks_table
    .style
    .format({
        'Количество клиентов': '{:,.0f}',
        'Клиенты с просмотрами/кликами без подписки': '{:,.0f}',
        'Доля клиентов, %': '{:.1f}%'
    })
)

In [ ]:
plt.figure(figsize=(10, 5))

bars = plt.bar(
    cohort_clicks['campaigns_cnt'].astype(str),
    cohort_clicks['click_pct']
)

for bar, pct in zip(bars, cohort_clicks['click_pct']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{pct:.1f}%',
        ha='center'
    )

plt.title('Доля клиентов с просмотрами/кликами оффера без подписки')
plt.xlabel('Количество кампаний')
plt.ylabel('Доля клиентов, %')
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()